In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
data=pd.read_csv('/kaggle/input/phishingemails/Phishing_Email.csv')
data.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
data['Email Type'] = data['Email Type'].astype('category')
print("各类别样本数：")
print(data['Email Type'].value_counts())
data.head()

各类别样本数：
Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64


,ID,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [3]:
#Training and Testing
np.random.seed(42)  
train_data = pd.DataFrame()
test_data = pd.DataFrame()

for email_type in data['Email Type'].unique():
    subset = data[data['Email Type'] == email_type].copy()
    indices = np.random.permutation(len(subset))
    n_total = len(subset)
    n_train = int(0.8 * n_total)
    
    train_indices = indices[:n_train]
    test_indices = indices[n_train:]
    
    train_subset = subset.iloc[train_indices].reset_index(drop=True)
    test_subset = subset.iloc[test_indices].reset_index(drop=True)
    
    train_data = pd.concat([train_data, train_subset], ignore_index=True)
    test_data = pd.concat([test_data, test_subset], ignore_index=True)

train_data = train_data.sample(frac=1, random_state=42).reset_index(drop=True)
test_data = test_data.sample(frac=1, random_state=42).reset_index(drop=True)

train_data.to_csv('train_dataset.csv', index=False)
test_data.to_csv('test_dataset.csv', index=False)

In [4]:
#Regulization
import re
from typing import List

def tokenizer(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', ' ', text)
    date_patterns = [
        r'\b\d{4}-\d{1,2}-\d{1,2}\b',
        r'\b\d{1,2}/\d{1,2}/\d{2,4}\b',
        r'\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{1,2},?\s+\d{4}\b',
        r'\b\d{1,2}(?:st|nd|rd|th)?\s+(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{4}\b'
    ]
    for pattern in date_patterns:
        text = re.sub(pattern, ' ', text)
    text = re.sub(r'\b\d+[.,]?\d*\b', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    filtered_tokens = [token for token in tokens if len(token) >= 2]
    
    return filtered_tokens

In [5]:
#Read the data
train_data=pd.read_csv("/kaggle/working/train_dataset.csv")
test_data=pd.read_csv("/kaggle/working/test_dataset.csv")
train_data = train_data['Email Text'].fillna('').astype(str)
test_data = test_data['Email Text'].fillna('').astype(str)
train_data.head()

0    howdy all,I have in my possession the followin...
1                                                empty
2    CNET Investor Dispatch\nQuote LookupEnter symb...
3    saiba como aumentar seu penis em até 7 cm saib...
4    File-traders in the crosshairsÂ Search\n\t    ...
Name: Email Text, dtype: object

In [6]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(
    tokenizer=tokenizer,         
    max_features=1000,
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.90
)
X_train_tfidf = vectorizer.fit_transform(train_data)
X_test_tfidf = vectorizer.transform(test_data)

feature_names = vectorizer.get_feature_names_out()

df_train_tfidf = pd.DataFrame(X_train_tfidf.toarray(), columns=feature_names)
df_test_tfidf = pd.DataFrame(X_test_tfidf.toarray(), columns=feature_names)

df_train_tfidf.to_csv('train_tfidf_features.csv', index=False)
df_test_tfidf.to_csv('test_tfidf_features.csv', index=False)

print("TF-IDF 向量化完成")
print(f"特征维度: {len(feature_names)}")
print(f"训练集形状: {df_train_tfidf.shape}")
print(f"测试集形状: {df_test_tfidf.shape}")

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TF-IDF 向量化完成
特征维度: 1000
训练集形状: (14919, 1000)
测试集形状: (3731, 1000)


In [7]:
# Label
train_data=pd.read_csv("/kaggle/working/train_dataset.csv")
test_data=pd.read_csv("/kaggle/working/test_dataset.csv")
df_train_tfidf['Label'] = train_data['Email Type'].values
df_test_tfidf['Label'] = test_data['Email Type'].values

df_train_tfidf.to_csv('train_tfidf_with_labels.csv', index=False)
df_test_tfidf.to_csv('test_tfidf_with_labels.csv', index=False)

print("已保存带标签的 TF-IDF 数据集")

已保存带标签的 TF-IDF 数据集
